In [ ]:
import os

# ─── PATHS ────────────────────────────────────────────────────────────────────
BASE_DIR        = os.path.dirname(os.path.abspath("/kaggle/working"))
DATA_DIR        = os.path.join(BASE_DIR, "data")
CHECKPOINTS_DIR = "/kaggle/working/checkpoints"
LOGS_DIR        = os.path.join(BASE_DIR, "logs")
RESULTS_DIR     = os.path.join(BASE_DIR, "results")

for d in [DATA_DIR, CHECKPOINTS_DIR, LOGS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── KAGGLE DATASET SLUGS ─────────────────────────────────────────────────────
KAGGLE_DATASETS = {
    "weapons":   "issaisasank/guns-object-detection",   # YOLO-format weapon images
    "violence":  "mohamedmustafa/real-life-violence-situations-dataset",
    "emotion":   "msambare/fer2013",                    # FER2013 facial expression
    "anomaly":   "odins0n/ucf-crime-dataset-small",     # UCF-Crime subset
}
# NOTE: set your Kaggle API credentials before running Part 1:
#   export KAGGLE_USERNAME=your_username
#   export KAGGLE_KEY=your_api_key
# OR place kaggle.json in ~/.kaggle/

# ─── MODEL CHECKPOINTS (output names) ─────────────────────────────────────────
WEAPON_BEST_MODEL    = "/kaggle/input/datasets/mfonisovic/weapons-best2/best (1).pt"
VIOLENCE_BEST_MODEL  = "/kaggle/input/datasets/mfonisovic/violence-best/violence_best.pt"
EMOTION_BEST_MODEL   = "/kaggle/input/datasets/mfonisovic/emotion-vit/emotion_vit_best.pt"
FUSION_BEST_MODEL    = "/kaggle/working/checkpoints/fusion_best.pt"

# ─── TRAINING HYPERPARAMETERS ─────────────────────────────────────────────────
SEED = 42

WEAPON_CFG = {
    "model":        "yolov10s.pt",   # nano — fastest, pretrained on COCO
    "epochs":       80,
    "imgsz":        416,
    "batch":        16,
    "lr0":          0.001,
    "patience":     15,             # early stopping
    "conf_thresh":  0.5,
    "iou_thresh":   0.45,
}

VIOLENCE_CFG = {
    "backbone":     "mobilenet_v2", # pretrained on ImageNet
    "seq_len":      32,             # frames per clip
    "frame_size":   (112, 112),
    "lstm_hidden":  256,
    "lstm_layers":  2,
    "dropout":      0.4,
    "epochs":       60,
    "batch":        8,
    "lr":           3e-4,
    "weight_decay": 1e-4,
    "patience":     12,
    "num_classes":  2,              # violent / non-violent
}

EMOTION_CFG = {
    "backbone":     "",  # pretrained on ImageNet
    "img_size":     (224, 224),
    "epochs":       50,
    "batch":        64,
    "lr":           1e-3,
    "weight_decay": 1e-4,
    "patience":     10,
    "num_classes":  7,              # FER2013: angry,disgust,fear,happy,sad,surprise,neutral
    "fear_class":   2,              # index of 'fear' in FER2013
}

FUSION_CFG = {
    "epochs":           40,
    "batch":            32,
    "lr":               1e-3,
    "weight_decay":     1e-4,
    "patience":         10,
    # Predetermined threat weights (priority order: violence>weapons>precursor>emotion)
    # These are used as INITIALISATION if learn_weights=True,
    # or as fixed values if learn_weights=False.
# Set True to let model learn weights; False = fixed
    "learn_weights": True,          # <-- important
    "threat_weights": {             # used for warm‑starting the MLP
        "violence": 0.40,
        "weapon": 0.30,
        "precursor": 0.20,
        "emotion": 0.10
    }
}

# ─── THREAT SCORING THRESHOLDS ────────────────────────────────────────────────
THREAT = {
    "low":    0.4,   # below → log only
    "medium": 0.5,   # above → alert dashboard
    # above medium → immediate alert
    "loitering_secs":   180,
    "concealment_secs":   5,
    "gaze_avoid_secs":   10,
    "proximity_meters":   2,
}

# ─── DEVICE ───────────────────────────────────────────────────────────────────
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Config] Using device: {DEVICE}")


In [ ]:
!pip install ultralytics torch torchvision tqdm scikit-learn opencv-python pandas matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms, datasets

In [ ]:
import os
import cv2

# Paths
src_root = "/kaggle/input/datasets/issaisasank/guns-object-detection"
src_images = os.path.join(src_root, "Images")
src_labels = os.path.join(src_root, "Labels")

dst_root = "/kaggle/working/guns_yolo"
dst_images = os.path.join(dst_root, "images")
dst_labels = os.path.join(dst_root, "labels")
os.makedirs(dst_images, exist_ok=True)
os.makedirs(dst_labels, exist_ok=True)

image_files = [f for f in os.listdir(src_images) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print(f"Total images: {len(image_files)}")

for img_file in image_files:
    # Symlink image
    src_img = os.path.join(src_images, img_file)
    dst_img = os.path.join(dst_images, img_file)
    if not os.path.exists(dst_img):
        os.symlink(src_img, dst_img)

    # Read image dimensions
    img = cv2.imread(src_img)
    if img is None:
        print(f"Warning: cannot read {img_file}, skipping")
        continue
    h, w = img.shape[:2]

    # Corresponding label file
    label_base = os.path.splitext(img_file)[0] + ".txt"
    src_label = os.path.join(src_labels, label_base)
    dst_label = os.path.join(dst_labels, label_base)
    if not os.path.exists(src_label):
        print(f"Warning: no label for {img_file}")
        continue

    with open(src_label, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    # First line is the count of guns
    if not lines:
        continue
    try:
        num_guns = int(lines[0])
    except ValueError:
        print(f"Warning: invalid count line in {src_label}: {lines[0]}")
        continue

    # Next 'num_guns' lines are coordinates (xmin ymin xmax ymax)
    yolo_lines = []
    for i in range(1, min(num_guns, len(lines)-1) + 1):
        coord_parts = lines[i].split()
        if len(coord_parts) != 4:
            print(f"Warning: invalid coordinates in {src_label}: {lines[i]}")
            continue
        try:
            x_min, y_min, x_max, y_max = map(int, coord_parts)
        except ValueError:
            print(f"Warning: non-integer coordinates in {src_label}: {lines[i]}")
            continue

        # Convert to YOLO normalized format
        box_w = x_max - x_min
        box_h = y_max - y_min
        x_center = (x_min + box_w / 2) / w
        y_center = (y_min + box_h / 2) / h
        width_norm = box_w / w
        height_norm = box_h / h

        # Only one class: gun → class_id = 0
        yolo_lines.append(f"0 {x_center:.6f} {y_center:.6f} {width_norm:.6f} {height_norm:.6f}")

    # Write converted labels
    if yolo_lines:
        with open(dst_label, 'w') as f:
            f.write("\n".join(yolo_lines))
    else:
        print(f"Warning: no valid guns for {img_file}")

print("Conversion complete!")
print(f"Images: {len(os.listdir(dst_images))}, Labels: {len(os.listdir(dst_labels))}")

In [ ]:
from sklearn.model_selection import train_test_split

all_images = [f for f in os.listdir(dst_images) if f.endswith(('.jpg', '.jpeg', '.png'))]
train_imgs, val_imgs = train_test_split(all_images, test_size=0.2, random_state=42)

# Create train/val subdirectories
for split in ['train', 'val']:
    os.makedirs(os.path.join(dst_root, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(dst_root, split, 'labels'), exist_ok=True)

def link_split(images, split):
    for f in images:
        src_img = os.path.join(dst_images, f)
        dst_img = os.path.join(dst_root, split, 'images', f)
        if not os.path.exists(dst_img):
            os.symlink(src_img, dst_img)
        lbl = os.path.splitext(f)[0] + '.txt'
        src_lbl = os.path.join(dst_labels, lbl)
        dst_lbl = os.path.join(dst_root, split, 'labels', lbl)
        if not os.path.exists(dst_lbl):
            os.symlink(src_lbl, dst_lbl)

link_split(train_imgs, 'train')
link_split(val_imgs, 'val')

print(f"Train: {len(train_imgs)}, Val: {len(val_imgs)}")

In [ ]:
import yaml

data_yaml = {
    'path': dst_root,
    'train': 'train/images',
    'val': 'val/images',
    'nc': 1,   # only one class: gun
    'names': ['gun']
}

yaml_path = "/kaggle/working/weapon_data_fixed.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f)
print(f"Created {yaml_path}")

In [ ]:
from ultralytics import YOLO

# Load a YOLOv10s model. This will download the weights if they don't exist.
model = YOLO(WEAPON_BEST_MODEL)

In [ ]:
results = model.train(
    data=yaml_path,      
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,            
    project="runs/detect",
    name="weapon_train"
)

In [ ]:
import glob
import os

# Find all train directories
train_dirs = glob.glob("/kaggle/working/runs/detect/weapon_train*")
if train_dirs:
    latest = max(train_dirs, key=os.path.getctime)  # or use numerical sort
    best_yolo10_model = os.path.join(latest, "weights", "best.pt")
    print(f"Latest best model: {best_yolo10_model}")
else:
    print("No training runs found.")

In [ ]:
best_yolo10_model = model.trainer.best  

In [ ]:
FEAR_IDX = EMOTION_CFG["fear_class"]
class EmotionViT(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        from transformers import ViTForImageClassification
        self.vit = ViTForImageClassification.from_pretrained(
            'google/vit-base-patch16-224-in21k',
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        hidden_size = self.vit.config.hidden_size
        self.vit.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        outputs = self.vit(pixel_values)
        features = outputs.logits   # after removing classifier
        return self.head(features)

    def fear_score(self, x):
        logits = self.forward(x)
        probs = F.softmax(logits, dim=-1)
        return probs[:, FEAR_IDX]   # FEAR_IDX = 2

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ViolenceFrameDataset(Dataset):
    """
    Loads pre-extracted frame sequences from:
      data/violence/frames/{split}/{violence|nonviolence}/{clip_id}/*.jpg
    Returns a tensor of shape (seq_len, C, H, W) and an integer label.
    """
    CLASSES = {"nonviolence": 0, "violence": 1}

    def __init__(self, frames_root: str, split: str = "train",
                 seq_len: int = 32, augment: bool = False):
        self.seq_len  = seq_len
        self.augment  = augment
        self.clips    = []   # list of (clip_dir_path, label)

        base = Path(frames_root) / split
        for cls_name, label in self.CLASSES.items():
            cls_dir = base / cls_name
            if not cls_dir.exists():
                continue
            for clip_dir in sorted(cls_dir.iterdir()):
                frames = sorted(clip_dir.glob("*.jpg"))
                if len(frames) >= 4:
                    self.clips.append((clip_dir, label))

        # ── transforms ────────────────────────────────────────────────────────
        self.base_tfm = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(VIOLENCE_CFG["frame_size"]),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225]),
        ])
        self.aug_tfm = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(VIOLENCE_CFG["frame_size"]),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.2, hue=0.05),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225]),
        ])

    def __len__(self): return len(self.clips)

    def __getitem__(self, idx):
        import cv2
        clip_dir, label = self.clips[idx]
        frames = sorted(clip_dir.glob("*.jpg"))
        # Sample seq_len frames uniformly
        indices = np.linspace(0, len(frames) - 1, self.seq_len, dtype=int)
        tfm = self.aug_tfm if self.augment else self.base_tfm
        tensors = []
        for i in indices:
            img = cv2.imread(str(frames[i]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            tensors.append(tfm(img))
        clip_tensor = torch.stack(tensors)  # (seq_len, C, H, W)
        return clip_tensor, label

#  MODEL: CNN-Feature-Extractor + Bi-LSTM + Self-Attention

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)  # *2 for bidirectional

    def forward(self, lstm_out):
        # lstm_out: (batch, seq, hidden*2)
        scores  = self.attn(lstm_out).squeeze(-1)       # (batch, seq)
        weights = F.softmax(scores, dim=-1)              # (batch, seq)
        context = (weights.unsqueeze(-1) * lstm_out).sum(dim=1)  # (batch, hidden*2)
        return context, weights


class ViolenceDetector(nn.Module):
    """
    Architecture:
      MobileNetV2 (pretrained, frozen first 10 layers) → feature per frame
      → Bi-LSTM(256, 2 layers)
      → Temporal Self-Attention
      → Classifier head
    """
    def __init__(self, num_classes: int = 2,
                 lstm_hidden: int = 256,
                 lstm_layers: int = 2,
                 dropout: float = 0.4):
        super().__init__()
        self.lstm_hidden = lstm_hidden

        # CNN backbone (MobileNetV2 pretrained on ImageNet) 
        backbone = models.mobilenet_v2(
            weights=models.MobileNet_V2_Weights.IMAGENET1K_V1
        )
        # Remove classifier; keep feature extraction
        self.cnn = backbone.features  # output: (batch, 1280, H//32, W//32)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # → (batch, 1280)
        self.feat_dim = 1280

        # Freeze early layers to preserve pretrained spatial features
        for i, layer in enumerate(self.cnn):
            if i < 10:
                for p in layer.parameters():
                    p.requires_grad = False

        #  Temporal encoder 
        self.lstm = nn.LSTM(
            input_size  = self.feat_dim,
            hidden_size = lstm_hidden,
            num_layers  = lstm_layers,
            batch_first = True,
            bidirectional = True,
            dropout     = dropout if lstm_layers > 1 else 0.0,
        )
        self.attention = TemporalAttention(lstm_hidden)

        #  Classifier 
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # x: (batch, seq_len, C, H, W)
        B, T, C, H, W = x.shape
        # Flatten batch and time for CNN
        x_flat  = x.view(B * T, C, H, W)
        feats   = self.pool(self.cnn(x_flat)).squeeze(-1).squeeze(-1)  # (B*T, 1280)
        feats   = feats.view(B, T, self.feat_dim)                       # (B, T, 1280)

        lstm_out, _ = self.lstm(feats)                                  # (B, T, H*2)
        context, attn_weights = self.attention(lstm_out)                # (B, H*2)
        logits  = self.classifier(context)                              # (B, num_classes)
        return logits, attn_weights

    def extract_features(self, x):
        """Return feature embedding (for fusion) and attention weights."""
        logits, attn = self.forward(x)
        probs = F.softmax(logits, dim=-1)
        return probs[:, 1], attn   # violence probability scalar per sample


In [ ]:
!pip install torch torchvision ultralytics tqdm scikit-learn
import os, sys, random, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (classification_report, roc_auc_score, f1_score,
                              average_precision_score, confusion_matrix)

os.makedirs("/kaggle/working/checkpoints", exist_ok = True)

sys.path.insert(0, "/kaggle/working")
#from config import (DATA_DIR, CHECKPOINTS_DIR, LOGS_DIR, RESULTS_DIR,
 #                   WEAPON_BEST_MODEL, VIOLENCE_BEST_MODEL,
  #                  EMOTION_BEST_MODEL, FUSION_BEST_MODEL,
   #                 WEAPON_CFG, VIOLENCE_CFG, EMOTION_CFG, FUSION_CFG,
    #                THREAT, DEVICE, SEED)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

device = torch.device(DEVICE)
WEAPON_BEST_MODEL = best_yolo10_model


#  MODULE LOADERS  (frozen — used for feature extraction only)

def load_violence_model():
    """Load the pretrained violence detector and freeze all parameters."""
    # Import from the training script
    sys.path.insert(0, os.path.dirname("/kaggle/working"))
    cfg  = VIOLENCE_CFG
    mdl  = ViolenceDetector(
        num_classes=cfg["num_classes"],
        lstm_hidden=cfg["lstm_hidden"],
        lstm_layers=cfg["lstm_layers"],
        dropout=0.0,   # no dropout at inference
    ).to(device)
    ckpt = torch.load(VIOLENCE_BEST_MODEL, map_location=device, weights_only=False)
    mdl.load_state_dict(ckpt["model_state"])
    mdl.eval()
    for p in mdl.parameters(): p.requires_grad = False
    print(f"  [Fusion] Violence model loaded (val F1={ckpt['val_f1']:.4f})")
    return mdl


def load_emotion_model():
    
    cfg  = EMOTION_CFG
    mdl  = EmotionViT(num_classes=cfg["num_classes"]).to(device)
    state_dict = torch.load(EMOTION_BEST_MODEL, map_location=device, weights_only=False)
    mdl.load_state_dict(state_dict)
 #   ckpt = torch.load(EMOTION_BEST_MODEL, map_location=device, weights_only=False)
#    mdl.load_state_dict(ckpt["model_state"])
    mdl.eval()
    for p in mdl.parameters(): p.requires_grad = False
    print(f"  [Fusion] Emotion model loaded (val F1-fear={ckpt['val_f1_fear']:.4f})")
    return mdl


def load_weapon_model():
    from ultralytics import YOLO
    mdl = YOLO(WEAPON_BEST_MODEL)
    print(f"  [Fusion] Weapon model loaded from {WEAPON_BEST_MODEL}")
    return mdl



#  FUSION DATASET  (uses pre-computed module scores from a manifest CSV)

class FusionScoreDataset(Dataset):
    """
    Each sample is a dict of confidence scores:
      violence_score, weapon_score, emotion_score, precursor_score
      → binary threat label (high/low)
    Scores are pre-computed by running the frozen modules over the clips
    (see precompute_module_scores() below).
    """
    def __init__(self, scores_file: str):
        import pandas as pd
        self.df = pd.read_csv(scores_file)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        feats = torch.tensor([
            row["violence_score"],
            row["weapon_score"],
            row["emotion_score"],
            row["precursor_score"],
        ], dtype=torch.float32)
        label = int(row["threat_label"])   # 1=threat, 0=no threat
        return feats, label



#  PRE-COMPUTE MODULE SCORES

@torch.no_grad()
def precompute_module_scores(violence_model, emotion_model, weapon_model,
                              frames_root: str, out_csv: str,
                              seq_len: int = 32):
    """
    Runs each frozen module over all clips in frames_root and saves
    a CSV of (clip_id, split, violence_score, weapon_score,
              emotion_score, precursor_score, threat_label).
    This decouples feature extraction from fusion training and allows
    the fusion CSV to be reused without re-running all three modules.
    """
    import cv2, pandas as pd
    from torchvision import transforms

    CLASSES = {"nonviolence": 0, "violence": 1}
    v_tfm   = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize(VIOLENCE_CFG["frame_size"]),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    e_tfm   = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize(EMOTION_CFG["img_size"]),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),

    ])

    rows = []
    for split in ["train", "val", "test"]:
        for cls_name, violence_label in CLASSES.items():
            cls_dir = Path(frames_root) / split / cls_name
            if not cls_dir.exists(): continue
            for clip_dir in tqdm(sorted(cls_dir.iterdir()),
                                 desc=f"{split}/{cls_name}"):
                frame_files = sorted(clip_dir.glob("*.jpg"))
                if not frame_files: continue

               # Violence score
                indices = np.linspace(0, len(frame_files)-1, seq_len, dtype=int)
                v_frames = []
                for i in indices:
                    img = cv2.imread(str(frame_files[i]))
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    v_frames.append(v_tfm(img))
                clip_t = torch.stack(v_frames).unsqueeze(0).to(device)  # (1,T,C,H,W)
                v_prob, _ = violence_model.extract_features(clip_t)
                v_score   = float(v_prob.item())

                #  Emotion score (avg fear score over middle 8 frames) 
                mid_idx = [int(x) for x in np.linspace(len(frame_files)//4,
                                                         3*len(frame_files)//4, 8, dtype=int)]
                e_frames = []
                for i in mid_idx:
                    img = cv2.imread(str(frame_files[i]))
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    e_frames.append(e_tfm(img))
                e_batch   = torch.stack(e_frames).to(device)
                e_score   = float(emotion_model.fear_score(e_batch).mean().item())

                #  Weapon score (YOLO on 4 keyframes) 
                w_frames = [str(frame_files[int(i)])
                            for i in np.linspace(0, len(frame_files)-1, 4, dtype=int)]
                w_results = weapon_model(w_frames,
                                        conf=WEAPON_CFG["conf_thresh"],
                                        verbose=False)
                # Max weapon confidence across all frames and all detections
                w_score = 0.0
                for r in w_results:
                    if r.boxes is not None and len(r.boxes):
                        confs = r.boxes.conf.cpu().numpy()
                        w_score = max(w_score, float(confs.max()))

                #  Precursor score (rule-based motion heuristic) 
                p_score = _compute_precursor_score(frame_files)

                #  Threat label  (violence or weapon present = threat) 
                threat_label = violence_label  # violence clips → threat

                rows.append({
                    "clip_id":        clip_dir.name,
                    "split":          split,
                    "class":          cls_name,
                    "violence_score": round(v_score, 5),
                    "weapon_score":   round(w_score, 5),
                    "emotion_score":  round(e_score, 5),
                    "precursor_score":round(p_score, 5),
                    "threat_label":   threat_label,
                })

    import pandas as pd
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"\n[Fusion] Scores saved → {out_csv}  ({len(rows)} clips)")


def _compute_precursor_score(frame_files: list, sample_n: int = 16) -> float:
    """
    Lightweight optical-flow-based precursor score.
    High motion variance with low overall displacement → suspicious lingering.
    Returns a score in [0, 1].
    """
    import cv2
    if len(frame_files) < 4:
        return 0.0
    indices  = np.linspace(0, len(frame_files)-1, min(sample_n, len(frame_files)), dtype=int)
    frames   = [cv2.cvtColor(cv2.imread(str(frame_files[i])), cv2.COLOR_BGR2GRAY)
                for i in indices]
    mags = []
    for i in range(len(frames) - 1):
        flow = cv2.calcOpticalFlowFarneback(
            frames[i], frames[i+1], None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0
        )
        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        mags.append(mag.mean())
    if not mags: return 0.0
    mags     = np.array(mags)
    variance = mags.std() / (mags.mean() + 1e-6)
    # Normalise to ~[0,1]; high variance relative to mean → suspicious
    score    = float(np.clip(variance / 3.0, 0.0, 1.0))
    return score


#  FUSION HEAD MODEL

    """
    Lightweight MLP that fuses four module confidence scores into a
    binary threat prediction.

    If config learn_weights=False:
        Uses fixed, predetermined weights (priority: violence>weapon>precursor>emotion).
        The weighted sum S_threat = w1*V + w2*W + w4*P + w3*F is fed to a
        sigmoid threshold — this is equivalent to a linear probit model.
    If config learn_weights=True:
        Initialises weights from the predetermined values, but allows
        the MLP to learn non-linear interactions.
    """
class ThreatFusionHead(nn.Module):
    def __init__(self, learn_weights: bool = False, init_weights: dict = None):
        super().__init__()
        self.learn_weights = learn_weights
        w = init_weights or {'violence':0.40, 'weapon':0.30, 'precursor':0.20, 'emotion':0.10}

        if not learn_weights:
            # Fixed weights (no training)
            self.register_buffer('weights', torch.tensor(
                [w['violence'], w['weapon'], w['precursor'], w['emotion']],
                dtype=torch.float32
            ))
        else:
            # Small MLP with 4 inputs → 1 output
            self.mlp = nn.Sequential(
                nn.Linear(4, 16),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(16, 8),
                nn.ReLU(),
                nn.Linear(8, 1)
            )
            # Warm‑start the first layer with the predetermined weights
            with torch.no_grad():
                init_w = torch.tensor([
                    w['violence'], w['weapon'], w['precursor'], w['emotion']
                ], dtype=torch.float32)
                # Set the first row of the weight matrix to init_w
                self.mlp[0].weight.data[0, :] = init_w
                # Small random noise for the remaining rows (15 rows because 4→16)
                self.mlp[0].weight.data[1:, :] = torch.randn(15, 4) * 0.01

    def forward(self, x):
        # x: (batch, 4) -> [violence, weapon, precursor, emotion]
        if not self.learn_weights:
            return (x * self.weights).sum(dim=-1, keepdim=True)   # raw logit
        else:
            return self.mlp(x)

    def threat_probability(self, x):
        return torch.sigmoid(self.forward(x))

    def get_threat_weights(self):
        """Return current effective weights for interpretability."""
        if not self.learn_weights:
            return self.weights.cpu().numpy()
        else:
            # Return first-layer weight magnitudes as proxy
            return self.mlp[0].weight.data.abs().mean(0).cpu().numpy()


#  TRAIN FUSION HEAD

def train_fusion_head(scores_csv: str):
    cfg = FUSION_CFG
    print("=" * 60)
    print("Part 3 — Multi-Modal Fusion Head Training")
    print(f"learn_weights={cfg['learn_weights']}  Device: {DEVICE}")
    print("=" * 60)

    import pandas as pd
    df_all = pd.read_csv(scores_csv)
    train_csv = os.path.join(DATA_DIR, "_fusion_train.csv")
    val_csv   = os.path.join(DATA_DIR, "_fusion_val.csv")
    test_csv  = os.path.join(DATA_DIR, "_fusion_test.csv")
    df_all[df_all.split == "train"].to_csv(train_csv, index=False)
    df_all[df_all.split == "val"  ].to_csv(val_csv,   index=False)
    df_all[df_all.split == "test" ].to_csv(test_csv,  index=False)

    train_ds = FusionScoreDataset(train_csv)
    val_ds   = FusionScoreDataset(val_csv)
    test_ds  = FusionScoreDataset(test_csv)
    print(f"  Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

    # Class-balanced sampler
    labels  = [train_ds[i][1] for i in range(len(train_ds))]
    counts  = np.bincount(labels)
    w_samp  = 1.0 / counts[labels]
    sampler = torch.utils.data.WeightedRandomSampler(w_samp, len(w_samp))

    train_loader = DataLoader(train_ds, batch_size=cfg["batch"],
                              sampler=sampler, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch"], shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=cfg["batch"], shuffle=False)


    
    model = ThreatFusionHead(
        learn_weights = cfg["learn_weights"],
        init_weights = cfg["threat_weights"]
    ).to(device)
    # Check for trainable parameters
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if not trainable_params:
        print("No trainable parameters – using fixed weights. Skipping training.")
        # Optionally evaluate directly
        return
    
    
    # ... rest of training loop ...
 #   model     = ThreatFusionHead(
 #       learn_weights=cfg["learn_weights"],
 #       init_weights =cfg["threat_weights"]
 #    ).to(device)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([counts[0]/counts[1]], dtype=torch.float32).to(device)
    )
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg["lr"],
                                  weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=5)

    best_f1    = 0.0
    no_improve = 0
    log_path   = os.path.join(LOGS_DIR, "fusion_training.csv")
    os.makedirs(LOGS_DIR, exist_ok=True)

    with open(log_path, "w") as log:
        log.write("epoch,tr_loss,tr_f1,vl_loss,vl_f1,vl_auc\n")

        for epoch in range(1, cfg["epochs"] + 1):
            # Train
            model.train()
            tr_losses, tr_preds, tr_labels = [], [], []
            for feats, lbls in tqdm(train_loader, desc="  Train", leave=False):
                feats = feats.to(device); lbls = lbls.float().to(device)
                optimizer.zero_grad()
                logits = model(feats).squeeze(-1)
                loss   = criterion(logits, lbls)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                tr_losses.append(loss.item())
                tr_preds.extend((torch.sigmoid(logits) > 0.5).long().cpu().tolist())
                tr_labels.extend(lbls.long().cpu().tolist())
            tr_f1 = f1_score(tr_labels, tr_preds, zero_division=0)

            # Validate
            model.eval()
            vl_losses, vl_preds, vl_labels, vl_probs = [], [], [], []
            with torch.no_grad():
                for feats, lbls in val_loader:
                    feats = feats.to(device); lbls = lbls.float().to(device)
                    logits = model(feats).squeeze(-1)
                    loss   = criterion(logits, lbls)
                    probs  = torch.sigmoid(logits)
                    vl_losses.append(loss.item())
                    vl_preds.extend((probs > 0.5).long().cpu().tolist())
                    vl_labels.extend(lbls.long().cpu().tolist())
                    vl_probs.extend(probs.cpu().tolist())
            vl_f1  = f1_score(vl_labels, vl_preds, zero_division=0)
            try:    vl_auc = roc_auc_score(vl_labels, vl_probs)
            except: vl_auc = 0.0
            scheduler.step(vl_f1)

            print(f"Epoch {epoch:3d}/{cfg['epochs']} | "
                  f"Tr Loss {np.mean(tr_losses):.4f} F1 {tr_f1:.4f} | "
                  f"Val Loss {np.mean(vl_losses):.4f} F1 {vl_f1:.4f} AUC {vl_auc:.4f}")
            log.write(f"{epoch},{np.mean(tr_losses):.5f},{tr_f1:.5f},"
                      f"{np.mean(vl_losses):.5f},{vl_f1:.5f},{vl_auc:.5f}\n")
            log.flush()

            if vl_f1 > best_f1:
                best_f1 = vl_f1; no_improve = 0
                torch.save({
                    "epoch":       epoch,
                    "model_state": model.state_dict(),
                    "val_f1":      vl_f1,
                    "val_auc":     vl_auc,
                    "config":      cfg,
                    "weights":     model.get_threat_weights().tolist(),
                }, FUSION_BEST_MODEL)
                print(f"  ✅  New best fusion model saved (F1={vl_f1:.4f})")
            else:
                no_improve += 1
                if no_improve >= cfg["patience"]:
                    print(f"  Early stopping at epoch {epoch}."); break

    # Test evaluation
    print("\n--- Final Fusion Test Evaluation ---")
    ckpt = torch.load(FUSION_BEST_MODEL, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    ts_preds, ts_labels, ts_probs = [], [], []
    with torch.no_grad():
        for feats, lbls in test_loader:
            feats = feats.to(device)
            probs = model.threat_probability(feats).squeeze(-1)
            ts_preds.extend((probs > 0.5).long().cpu().tolist())
            ts_labels.extend(lbls.tolist())
            ts_probs.extend(probs.cpu().tolist())
    print(classification_report(ts_labels, ts_preds,
                                 target_names=["No Threat","Threat"], digits=4))
    print("Confusion Matrix:\n", confusion_matrix(ts_labels, ts_preds))
    print(f"AUC-ROC: {roc_auc_score(ts_labels, ts_probs):.4f}")
    print(f"PR-AUC : {average_precision_score(ts_labels, ts_probs):.4f}")
    print(f"\nFinal Threat Weights: {ckpt['weights']}")


#  MAIN 
if __name__ == "__main__":
    scores_csv = os.path.join(DATA_DIR, "fusion_scores.csv")

    if not os.path.exists(scores_csv):
        print("[Fusion] Pre-computing module scores …")
        # The module imports below use relative names for clarity;
        # adjust if files are in a different directory.
        import importlib.util as ilu

        def load_module(file_path, module_name):
            spec = ilu.spec_from_file_location(module_name, file_path)
            mod  = ilu.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod

        base = os.path.dirname("/kaggle/working")
#        mod_v = load_module(os.path.join(base, "02b_train_violence_DEVICE_B.py"),
#                            "train_02b_train_violence_DEVICE_B")
#        mod_e = load_module(os.path.join(base, "02c_train_emotion_DEVICE_C.py"),
#                            "train_02c_train_emotion_DEVICE_C")

        violence_model = ViolenceDetector(
            num_classes=VIOLENCE_CFG["num_classes"],
            lstm_hidden=VIOLENCE_CFG["lstm_hidden"],
            lstm_layers=VIOLENCE_CFG["lstm_layers"],
            dropout=0.0,
        ).to(device)
        ckpt = torch.load(VIOLENCE_BEST_MODEL, map_location=device, weights_only=False)
        violence_model.load_state_dict(ckpt["model_state"])
        violence_model.eval()
        for p in violence_model.parameters(): p.requires_grad = False

        emotion_model = EmotionViT(
            num_classes=EMOTION_CFG["num_classes"]
        ).to(device)
        ckpt = torch.load(EMOTION_BEST_MODEL, map_location=device, weights_only=False)
        emotion_model.load_state_dict(ckpt)
        emotion_model.eval()
        for p in emotion_model.parameters(): p.requires_grad = False


        from ultralytics import YOLO
        weapon_model = YOLO(WEAPON_BEST_MODEL)

        frames_root = "/kaggle/input/notebooks/mfonisovic/notebook270892fa30/data/violence/frames"
        precompute_module_scores(
            violence_model, emotion_model, weapon_model,
            frames_root, scores_csv
        )
    else:
        print(f"[Fusion] Using cached scores: {scores_csv}")

    train_fusion_head(scores_csv)


In [ ]:
import os, sys, json, time
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm

sys.path.insert(0, os.path.dirname("/Kaggle/working"))


import importlib.util as ilu
def _load_mod(path, name):
    spec = ilu.spec_from_file_location(name, path)
    m    = ilu.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m

base    = os.path.dirname("/kaggle/working")
#mod_v   = _load_mod(os.path.join(base, "02b_train_violence_DEVICE_B.py"), "mod_v")
#mod_e   = _load_mod(os.path.join(base, "02c_train_emotion_DEVICE_C.py"),  "mod_e")
#mod_f   = _load_mod(os.path.join(base, "03_train_fusion.py"),              "mod_f")

os.makedirs(RESULTS_DIR, exist_ok=True)
device = torch.device(DEVICE)


#
#  METRIC HELPERS

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve,
)

def compute_binary_metrics(y_true, y_pred, y_prob,
                            pos_label: int = 1,
                            report_name: str = "") -> dict:
    """Comprehensive binary classification metrics."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    metrics = {
        "accuracy":          accuracy_score(y_true, y_pred),
        "precision":         precision_score(y_true, y_pred, zero_division=0),
        "recall":            recall_score   (y_true, y_pred, zero_division=0),
        "f1":                f1_score       (y_true, y_pred, zero_division=0),
        "specificity":       tn / (tn + fp + 1e-9),
        "false_positive_rate": fp / (fp + tn + 1e-9),
        "false_negative_rate": fn / (fn + tp + 1e-9),
        "auc_roc":           roc_auc_score  (y_true, y_prob),
        "avg_precision":     average_precision_score(y_true, y_prob),   # PR-AUC
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
    }
    if report_name:
        print(f"\n{'='*50}")
        print(f"  {report_name}")
        print(f"{'='*50}")
        for k, v in metrics.items():
            if isinstance(v, float):
                print(f"  {k:28s}: {v:.4f}")
            else:
                print(f"  {k:28s}: {v}")
    return metrics


def compute_detection_fps(model_fn, sample_input, n_iters: int = 50) -> float:
    """Measure inference FPS for a model forward pass."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_iters):
            model_fn(sample_input)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return n_iters / elapsed



#  LOAD ALL MODELS

def load_all_models():
    print("[Eval] Loading models …")
    from ultralytics import YOLO

    v_model = ViolenceDetector(
        num_classes=VIOLENCE_CFG["num_classes"],
        lstm_hidden=VIOLENCE_CFG["lstm_hidden"],
        lstm_layers=VIOLENCE_CFG["lstm_layers"],
        dropout=0.0,
    ).to(device)
    v_model.load_state_dict(
        torch.load(VIOLENCE_BEST_MODEL, map_location=device, weights_only=False)["model_state"])
    v_model.eval()

    e_model = EmotionViT(
        num_classes=EMOTION_CFG["num_classes"]
    ).to(device)
    e_model.load_state_dict(
        torch.load(EMOTION_BEST_MODEL, map_location=device, weights_only=False))
    e_model.eval()

    w_model = YOLO(WEAPON_BEST_MODEL)
    f_model = ThreatFusionHead(
        learn_weights=FUSION_CFG.get("learn_weights", False),
        init_weights=FUSION_CFG.get("threat_weights", {
           'violence': 0.40, 'weapon': 0.30,
           'precursor': 0.20, 'emotion': 0.10
        }),
    ).to(device)

    if os.path.exists(FUSION_BEST_MODEL):
        checkpoint = torch.load(FUSION_BEST_MODEL, map_location=device, weights_only=False)
        f_model.load_state_dict(checkpoint["model_state"])
        print("[Eval] Loaded fusion checkpoint")
    else:
        print(f"[Eval] No checkpoint at {FUSION_BEST_MODEL} – using fixed weights (no training)")
    f_model.eval()

#    f_model = ThreatFusionHead(
#        learn_weights=FUSION_CFG["learn_weights"],
#        init_weights =FUSION_CFG["threat_weights"],
#    ).to(device)
#    f_model.load_state_dict(
#        torch.load(FUSION_BEST_MODEL, map_location=device)["model_state"])
#    f_model.eval()

    print("[Eval] All models loaded.")
    return v_model, e_model, w_model, f_model



#  MODULE-SPECIFIC EVALUATIONS

def evaluate_violence_module(v_model) -> dict:
    from torch.utils.data import DataLoader
    frames_dir   = "/kaggle/input/notebooks/mfonisovic/notebook270892fa30/data/violence/frames"
    test_ds      = ViolenceFrameDataset(frames_dir, "test",
                                              VIOLENCE_CFG["seq_len"])
    test_loader  = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=2)
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for clips, lbls in tqdm(test_loader, desc="[Eval] Violence"):
            clips  = clips.to(device)
            logits, _ = v_model(clips)
            probs  = F.softmax(logits, dim=-1)[:, 1]
            all_preds.extend((probs > 0.5).long().cpu().tolist())
            all_labels.extend(lbls.tolist())
            all_probs.extend(probs.cpu().tolist())

    # FPS
    sample = torch.randn(1, VIOLENCE_CFG["seq_len"], 3,
                         *VIOLENCE_CFG["frame_size"]).to(device)
    fps    = compute_detection_fps(lambda x: v_model(x), sample)

    metrics = compute_binary_metrics(all_labels, all_preds, all_probs,
                                     report_name="Violence Detection Module")
    metrics["fps"] = fps
    return metrics


def evaluate_emotion_module(e_model) -> dict:
    from torch.utils.data import DataLoader
    from torchvision import datasets, transforms
    test_tfm  = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize(EMOTION_CFG["img_size"]),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    ])
    test_ds    = datasets.ImageFolder(
        "/kaggle/input/datasets/msambare/fer2013/test", transform=test_tfm)
    test_loader= DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
    fear_idx   = EMOTION_CFG["fear_class"]
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(test_loader, desc="[Eval] Emotion"):
            imgs  = imgs.to(device)
            probs = F.softmax(e_model(imgs), dim=-1)
            fear_prob = probs[:, fear_idx].cpu()
            pred_cls  = probs.argmax(dim=-1).cpu()
            # Binary: fear vs non-fear
            all_preds.extend((pred_cls == fear_idx).long().tolist())
            all_labels.extend((lbls == fear_idx).long().tolist())
            all_probs.extend(fear_prob.tolist())

    sample  = torch.randn(1, 3, *EMOTION_CFG["img_size"]).to(device)
    fps     = compute_detection_fps(lambda x: e_model(x), sample)
    metrics = compute_binary_metrics(all_labels, all_preds, all_probs,
                                     report_name="Emotion / Fear Module")
    metrics["fps"] = fps
    return metrics


def evaluate_weapon_module(w_model) -> dict:
    from ultralytics import YOLO
    import yaml
    data_yaml = yaml_path
    print("\n[Eval] Weapon Detection Module")
    val_res   = w_model.val(data=data_yaml,
                             conf=WEAPON_CFG["conf_thresh"],
                             iou=WEAPON_CFG["iou_thresh"],
                             verbose=False)
    metrics = {
        "mAP@0.5":     float(val_res.box.map50),
        "mAP@0.5:0.95":float(val_res.box.map),
        "precision":   float(val_res.box.mp),
        "recall":      float(val_res.box.mr),
    }
    print(f"  mAP@0.5     : {metrics['mAP@0.5']:.4f}")
    print(f"  mAP@0.5:0.95: {metrics['mAP@0.5:0.95']:.4f}")
    print(f"  Precision   : {metrics['precision']:.4f}")
    print(f"  Recall      : {metrics['recall']:.4f}")
    return metrics


def evaluate_fusion_system(f_model) -> dict:
    """Evaluate the full fused system on the cached scores CSV."""
    scores_csv = os.path.join(DATA_DIR, "fusion_scores.csv")
    if not os.path.exists(scores_csv):
        print("[Eval] ⚠  fusion_scores.csv not found. Skipping fusion eval.")
        return {}
    test_ds    = FusionScoreDataset(scores_csv)
    # Filter to test set via pandas
    import pandas as pd
    df = pd.read_csv(scores_csv)
    df_test = df[df.split == "test"].reset_index(drop=True)
    from torch.utils.data import TensorDataset, DataLoader
    feats  = torch.tensor(
        df_test[["violence_score","weapon_score","emotion_score","precursor_score"]].values,
        dtype=torch.float32
    )
    labels = torch.tensor(df_test["threat_label"].values, dtype=torch.long)
    ds     = TensorDataset(feats, labels)
    loader = DataLoader(ds, batch_size=64, shuffle=False)

    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for f, l in loader:
            f = f.to(device)
            p = f_model.threat_probability(f).squeeze(-1)
            all_preds.extend((p > 0.5).long().cpu().tolist())
            all_labels.extend(l.tolist())
            all_probs.extend(p.cpu().tolist())

    metrics = compute_binary_metrics(all_labels, all_preds, all_probs,
                                     report_name="Full Fusion System")

    # Threat-level breakdown
    probs_arr = np.array(all_probs)
    metrics["low_threat_pct"]    = float((probs_arr < THREAT["low"]).mean())
    metrics["medium_threat_pct"] = float(((probs_arr >= THREAT["low"]) &
                                          (probs_arr < THREAT["medium"])).mean())
    metrics["high_threat_pct"]   = float((probs_arr >= THREAT["medium"]).mean())
    return metrics



#  ABLATION STUDY

def run_ablation_study(f_model) -> dict:
    """
    Test fusion with each module zeroed out to measure its marginal contribution.
    Runs on the test split of the fusion scores CSV.
    """
    import pandas as pd
    scores_csv = os.path.join(DATA_DIR, "fusion_scores.csv")
    if not os.path.exists(scores_csv):
        return {}

    df     = pd.read_csv(scores_csv)
    df_t   = df[df.split == "test"].reset_index(drop=True)
    feats  = df_t[["violence_score","weapon_score","emotion_score","precursor_score"]].values
    labels = df_t["threat_label"].values

    results = {}
    combos  = {
        "full":             [0,1,2,3],
        "no_violence":      [1,2,3],
        "no_weapon":        [0,2,3],
        "no_emotion":       [0,1,3],
        "no_precursor":     [0,1,2],
        "violence_only":    [0],
        "weapon_only":      [1],
    }
    print("\n--- Ablation Study ---")
    for name, cols in combos.items():
        feat_ablated = feats.copy()
        for col in [0,1,2,3]:
            if col not in cols:
                feat_ablated[:, col] = 0.0
        f_t = torch.tensor(feat_ablated, dtype=torch.float32).to(device)
        with torch.no_grad():
            probs = f_model.threat_probability(f_t).squeeze(-1).cpu().numpy()
        preds = (probs > 0.5).astype(int)
        f1  = f1_score(labels, preds, zero_division=0)
        try:    auc = roc_auc_score(labels, probs)
        except: auc = 0.0
        results[name] = {"f1": round(f1,4), "auc": round(auc,4)}
        print(f"  {name:20s}  F1={f1:.4f}  AUC={auc:.4f}")
    return results



#  VISUALISATION

def save_evaluation_plots(v_metrics, e_metrics, f_metrics, ablation):
    """Save ROC, PR, confusion matrices, ablation bar chart."""
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from sklearn.metrics import roc_curve, precision_recall_curve
    except ImportError:
        print("[Eval] matplotlib not available — skipping plots.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("ProactiveGuard — Module Comparison", fontsize=14)

    #  Ablation bar chart 
    ax   = axes[0]
    names = list(ablation.keys())
    f1s  = [ablation[n]["f1"]  for n in names]
    aucs = [ablation[n]["auc"] for n in names]
    x    = np.arange(len(names))
    ax.bar(x - 0.2, f1s,  0.4, label="F1",     color="steelblue")
    ax.bar(x + 0.2, aucs, 0.4, label="AUC-ROC", color="coral")
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    ax.set_ylim(0, 1); ax.set_title("Ablation Study"); ax.legend()

    #  Module F1 / AUC comparison 
    ax     = axes[1]
    mods   = ["Violence\nModule", "Emotion\nModule", "Fusion\nSystem"]
    m_f1s  = [v_metrics.get("f1", 0), e_metrics.get("f1", 0), f_metrics.get("f1", 0)]
    m_aucs = [v_metrics.get("auc_roc", 0), e_metrics.get("auc_roc", 0), f_metrics.get("auc_roc", 0)]
    x      = np.arange(len(mods))
    ax.bar(x - 0.2, m_f1s,  0.4, label="F1",     color="steelblue")
    ax.bar(x + 0.2, m_aucs, 0.4, label="AUC-ROC", color="coral")
    ax.set_xticks(x); ax.set_xticklabels(mods)
    ax.set_ylim(0, 1); ax.set_title("Module Performance"); ax.legend()

    #  False Positive Rate comparison 
    ax    = axes[2]
    mods  = ["Violence", "Emotion", "Fusion"]
    fprs  = [v_metrics.get("false_positive_rate", 0),
             e_metrics.get("false_positive_rate", 0),
             f_metrics.get("false_positive_rate", 0)]
    bars  = ax.bar(mods, fprs, color=["steelblue","coral","seagreen"])
    ax.set_ylim(0, 0.5); ax.set_title("False Positive Rate (lower = better)")
    ax.set_ylabel("FPR")
    for bar, val in zip(bars, fprs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", fontsize=9)

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, "evaluation_report.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n[Eval] Plots saved → {out_path}")



#  PROACTIVE LEAD-TIME SIMULATION
def simulate_lead_time(f_model, n_samples: int = 200) -> dict:
    """
    Simulates lead-time analysis: given a synthetic timeline where a threat
    escalates over T frames, measure how many frames before the threshold
    the system triggers.
    """
    T = 100
    lead_times = []
    for _ in range(n_samples):
        # Simulate a gradually escalating threat
        v_scores = np.clip(np.linspace(0.0, 0.9, T) + np.random.randn(T)*0.05, 0, 1)
        w_scores = np.clip(np.linspace(0.0, 0.6, T) + np.random.randn(T)*0.05, 0, 1)
        e_scores = np.clip(np.linspace(0.0, 0.5, T) + np.random.randn(T)*0.05, 0, 1)
        p_scores = np.clip(np.linspace(0.0, 0.4, T) + np.random.randn(T)*0.05, 0, 1)
        true_incident = T - 1   # incident at last frame

        feats = torch.tensor(
            np.stack([v_scores, w_scores, e_scores, p_scores], axis=1),
            dtype=torch.float32
        ).to(device)
        with torch.no_grad():
            probs = f_model.threat_probability(feats).squeeze(-1).cpu().numpy()

        triggered = np.where(probs >= THREAT["medium"])[0]
        if len(triggered) > 0:
            lead_times.append(true_incident - triggered[0])

    # Always return valid numbers
    if lead_times:
        results = {
            "lead_time_mean": float(np.mean(lead_times)),
            "lead_time_std": float(np.std(lead_times)),
            "detection_rate": len(lead_times) / n_samples,
        }
    else:
        results = {
            "lead_time_mean": 0.0,
            "lead_time_std": 0.0,
            "detection_rate": 0.0,
        }
    print(f"\n--- Proactive Lead-Time Analysis ---")
    print(f"  Detection Rate  : {results['detection_rate']:.3f}")
    print(f"  Avg Lead Time   : {results['lead_time_mean']:.1f} frames")
    print(f"  Std Lead Time   : {results['lead_time_std']:.1f} frames")
    return results




# MAIN
if __name__ == "__main__":
    print("=" * 60)
    print("Part 4 — ProactiveGuard Full Evaluation Suite")
    print("=" * 60)

    v_model, e_model, w_model, f_model = load_all_models()

    v_metrics = evaluate_violence_module(v_model)
    e_metrics = evaluate_emotion_module(e_model)
    w_metrics = evaluate_weapon_module(w_model)
    f_metrics = evaluate_fusion_system(f_model)
    ablation  = run_ablation_study(f_model)
    lead_time = simulate_lead_time(f_model)

    # ── Print final summary ────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("SUMMARY — ProactiveGuard Evaluation")
    print("="*60)
    print(f"{'Module':<25} {'F1':>8} {'AUC':>8} {'FPR':>8} {'FPS':>8}")
    print("-"*60)
    print(f"{'Violence Detector':<25} {v_metrics.get('f1',0):>8.4f}"
          f" {v_metrics.get('auc_roc',0):>8.4f}"
          f" {v_metrics.get('false_positive_rate',0):>8.4f}"
          f" {v_metrics.get('fps',0):>8.1f}")
    print(f"{'Emotion / Fear':<25} {e_metrics.get('f1',0):>8.4f}"
          f" {e_metrics.get('auc_roc',0):>8.4f}"
          f" {e_metrics.get('false_positive_rate',0):>8.4f}"
          f" {e_metrics.get('fps',0):>8.1f}")
    print(f"{'Weapon Detector':<25} {'N/A':>8} {'N/A':>8} {'N/A':>8} {'N/A':>8}")
    print(f"  mAP@0.5={w_metrics.get('mAP@0.5',0):.4f}  "
          f"P={w_metrics.get('precision',0):.4f}  "
          f"R={w_metrics.get('recall',0):.4f}")
    print("-"*60)
    print(f"{'FUSION SYSTEM':<25} {f_metrics.get('f1',0):>8.4f}"
          f" {f_metrics.get('auc_roc',0):>8.4f}"
          f" {f_metrics.get('false_positive_rate',0):>8.4f}")

    report = {
        "violence_module": v_metrics,
        "emotion_module":  e_metrics,
        "weapon_module":   w_metrics,
        "fusion_system":   f_metrics,
        "ablation_study":  ablation,
        "lead_time":       lead_time,
    }
    report_path = os.path.join(RESULTS_DIR, "evaluation_report.json")
    with open(report_path, "w") as fp:
        json.dump(report, fp, indent=2)
    print(f"\n Full report saved → {report_path}")

    save_evaluation_plots(v_metrics, e_metrics, f_metrics, ablation)


In [ ]:
import os

results_dir = "/kaggle/results"
if os.path.exists(results_dir):
    print("Files in /kaggle/results:")
    for f in os.listdir(results_dir):
        print(f"  {f}")
else:
    print("Directory /kaggle/results not found. Trying /kaggle/working/results...")
    results_dir = "/kaggle/working/results"
    if os.path.exists(results_dir):
        for f in os.listdir(results_dir):
            print(f"  {f}")
    else:
        print("Results directory not found.")

In [ ]:
import shutil
shutil.copy('/kaggle/results/evaluation_report.json', '/kaggle/working/')
shutil.copy('/kaggle/results/evaluation_report.png', '/kaggle/working/')
print("Files copied to /kaggle/working. Refresh the file browser.")